In [1]:
# from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.document_loaders import TextLoader
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_core.documents import Document

from langchain_chroma import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
load_dotenv()

True

In [3]:
from dotenv import load_dotenv
import os

In [4]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
# GROQ_API_KEY

In [5]:
llm = ChatGroq(  
model="llama-3.3-70b-versatile",  
temperature=0.0,  
max_retries=2, 
)

In [ ]:
llm.invoke("what is deep learning in 10 words")

In [8]:
db_path=os.getenv("DB_PATH")
print(db_path)
kb_path=os.getenv("KB_PATH")
print(kb_path)
vector_store_path=os.getenv("VECTOR_STORE_PATH")
print(vector_store_path)

data/database/it_support.db
data/knowledge_base
vector_store


In [ ]:
"""
Mandatory tools:
1. retrieve_troubleshooting_steps
2. get_user_profile
3. get_device_status
4. check_known_incidents
5. run_diagnostic_check
6. get_ticket_details
7. create_resolution_plan
"""

In [9]:
import os
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def load_documents(folder_path=kb_path):
    docs = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".md"):   # adjust extension if needed
            file_path = os.path.join(folder_path, filename)
            loader = TextLoader(file_path, encoding="utf-8")
            loaded_docs = loader.load()
            # normalize metadata for each doc
            for i, doc in enumerate(loaded_docs, start=1):
                source_path = Path(file_path)
                doc.metadata = {
                    "source_file": source_path.name,
                    "issue_domain": source_path.stem,
                    "page_number": str(i),
                }
                docs.append(doc)
    return docs


In [10]:
docs = load_documents(kb_path)

In [11]:
docs

[Document(metadata={'source_file': 'email_outlook_troubleshooting_guide.md', 'issue_domain': 'email_outlook_troubleshooting_guide', 'page_number': '1'}, page_content='# Outlook and Email Troubleshooting Guide\n**Document ID:** IT-KB-EMAIL-001\n**Owner:** Messaging Support\n**Effective Date:** 2026-01-01\n\n## 1. Common Symptoms\nEmployees may report Outlook not syncing, email stuck in outbox, missing emails, repeated password prompts, calendar not updating, or mobile email issues.\n\n## 2. First-Level Checks\nCheck:\n- Internet connectivity.\n- Outlook client is open and not in offline mode.\n- Mailbox quota is not exceeded.\n- Microsoft 365 service health is normal.\n- User account is not locked.\n- Device date and time are correct.\n\n## 3. Outlook Not Syncing\nIf Outlook is not syncing:\n1. Confirm whether webmail works.\n2. Restart Outlook.\n3. Check offline mode.\n4. Check mailbox quota.\n5. Create a new Outlook profile if issue persists.\n6. Escalate to Messaging Support if webma

In [12]:
def split_and_set_metadata(docs, chunk_size=900, chunk_overlap=120):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunk_documents = text_splitter.split_documents(docs)

    chunks = []
    for i, chunk_document in enumerate(chunk_documents, start=1):
        metadata = dict(chunk_document.metadata)
        metadata["chunk_id"] = f"chunk_{i}"
        chunks.append(Document(page_content=chunk_document.page_content, metadata=metadata))
    return chunks


In [13]:

chunks = split_and_set_metadata(docs)

print(f"Loaded {len(docs)} docs.")
print(f"Total chunks: {len(chunks)}")
print(chunks[0].metadata)

Loaded 6 docs.
Total chunks: 14
{'source_file': 'email_outlook_troubleshooting_guide.md', 'issue_domain': 'email_outlook_troubleshooting_guide', 'page_number': '1', 'chunk_id': 'chunk_1'}


In [14]:
if docs:
    print(f"Loaded {len(docs)} pages from File.\n")
    print("First page content:\n", docs[0].page_content[:500], "...")
    print("\nMetadata:", docs[3].metadata)
    print("\nMetadata keys:", docs[0].metadata.keys())

Loaded 6 pages from File.

First page content:
 # Outlook and Email Troubleshooting Guide
**Document ID:** IT-KB-EMAIL-001
**Owner:** Messaging Support
**Effective Date:** 2026-01-01

## 1. Common Symptoms
Employees may report Outlook not syncing, email stuck in outbox, missing emails, repeated password prompts, calendar not updating, or mobile email issues.

## 2. First-Level Checks
Check:
- Internet connectivity.
- Outlook client is open and not in offline mode.
- Mailbox quota is not exceeded.
- Microsoft 365 service health is normal.
- Us ...

Metadata: {'source_file': 'password_reset_guide.md', 'issue_domain': 'password_reset_guide', 'page_number': '1'}

Metadata keys: dict_keys(['source_file', 'issue_domain', 'page_number'])


In [15]:
persist_directory=os.getenv("VECTOR_STORE_PATH")
persist_directory

'vector_store'

In [16]:
import chromadb
chromadb

<module 'chromadb' from 'c:\\Training\\Assignments\\AI_Training_Batch_May_2026\\submissions\\project-build\\langgraph-application\\Mohammad-Zaid\\it_troubleshooting_agent\\.venv\\Lib\\site-packages\\chromadb\\__init__.py'>

In [17]:
chromadb_client = chromadb.PersistentClient(
    path=persist_directory
)
chromadb_client

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [18]:
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [23]:
chunk_collection_name="issue-chunks"

In [ ]:

vector_db = Chroma(collection_name=chunk_collection_name,
                   collection_metadata={"hnsw:space": "cosine"}, 
                   embedding_function=embedding_model,
                   client=chromadb_client, 
                   persist_directory=persist_directory
                   )

In [ ]:
# def add_chunks_to_vector_db(chunks, vector_db):
vector_db._collection.count()

vector_db.add_documents(
        documents=chunks,
        ids=[chunk.metadata["chunk_id"] for chunk in chunks],
        )
        

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
# Test retrieval
query = "Company maximum attachment size is?"
retrieved_chunks = retriever.invoke(query)

In [ ]:
context = "\n\n".join([
    f"Source: {chunk.metadata.get('source_file')} | "
    f"issue: {chunk.metadata.get('policy_domain')} | "
    f"Page: {chunk.metadata.get('page_number')} | "
    f"Chunk: {chunk.metadata.get('chunk_id')}\n"
    f"{chunk.page_content}"
    for chunk in retrieved_chunks
])


In [ ]:
from langchain.tools.retriever import create_retriever_tool

In [ ]:
retrieve_troubleshooting_steps = create_retriever_tool(
    retriever=retriever,
    name="retrieve_troubleshooting_steps",
    description="Search and return information retrieve troubleshooting steps."
)

In [ ]:

"""
Mandatory tools:
1. retrieve_troubleshooting_steps
2. get_user_profile
3. get_device_status
4. check_known_incidents
5. run_diagnostic_check
6. get_ticket_details
7. create_resolution_plan
"""

### SQL Tool

In [25]:
db_path=os.getenv("DB_PATH")
db_path

'data/database/it_support.db'

In [26]:
import sqlite3

In [ ]:
def db_connection():
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn


def run_query(query: str, params: tuple = ()):
    conn = db_connection()
    
    try:
        cursor = conn.cursor()
        cursor.execute(query, params)        
        rows = cursor.fetchall()
        res = [dict(row) for row in rows]

        if not res:
            return "No records were found in Database."
        return res

    except Exception as e:
        return f"Error in Database: {str(e)}"
    
    finally:
        conn.close()

In [30]:
conn = db_connection()
conn

In [ ]:
schema="""
CREATE TABLE devices (
  device_id TEXT PRIMARY KEY,
  user_id TEXT,
  device_type TEXT,
  os TEXT,
  compliance_status TEXT,
  vpn_client_version TEXT,
  disk_free_percent INTEGER,
  cpu_usage_percent INTEGER,
  memory_usage_percent INTEGER,
  last_seen TEXT,
  FOREIGN KEY (user_id) REFERENCES users (user_id)
)"""

In [ ]:
def get_user_profile(user_id):
    """
    gets the user profile
    Arg:
    user_id:(str)
    
    Output:
    User Profile
    """
    query="""
    SELECT device_id, device_type, os, compliance_status, vpn_client_version, disk_free_percent, cpu_usage_percent, memory_usage_percent, last_seen
    FROM devices
    WHERE user_id = ?
    """
    res = run_query(query, (user_id,))
    return res

In [31]:
query="""
    SELECT device_id, device_type, os, compliance_status, vpn_client_version, disk_free_percent, cpu_usage_percent, memory_usage_percent, last_seen
    FROM devices
    WHERE user_id = ?
    """

In [38]:
res = run_query(query, ('USR-1001',))
res

[{'device_id': 'DEV-2001',
  'device_type': 'Laptop',
  'os': 'Windows 11',
  'compliance_status': 'Compliant',
  'vpn_client_version': '5.9',
  'disk_free_percent': 22,
  'cpu_usage_percent': 35,
  'memory_usage_percent': 62,
  'last_seen': '2026-06-12 08:45:00'}]